# 03. Final Evaluation on Official UNSW-NB15 Test Set (v2)
**NetSentinel Intrusion Detection System**

In accordance with strict ML evaluation standards:
- The winning model was selected purely based on cross-validation on the training set.
- The official test set (`clean_test.csv`, 59,543 records) was held out unseen until now.
- In this notebook, we run the single official test set evaluation and inspect:
  1. Macro F1, Accuracy, Attack Recall, and False Alarm Rate (FAR)
  2. Per-class Precision, Recall, and F1 scores
  3. The 6x6 Confusion Matrix
  4. Inspection of False Positives (Normal traffic flagged as attack) and False Negatives (Missed attacks)


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import sys
sys.path.append('../../')

from ml.src.config import CLASSES, NORMAL_CLASS, FINAL_METRICS_PATH, MODEL_ARTIFACT_PATH


## 1. Load Final Metrics JSON & Artifacts

In [ ]:
with open('../../ml/results/metrics/final_metrics.json', 'r') as f:
    metrics_data = json.load(f)

print('Evaluated Model:', metrics_data['primary_model'])
print('Model Version:  ', metrics_data['model_version'])
print('Agreement Rate: ', f"{metrics_data['model_agreement_rate']*100:.2f}%")
print('\nOverall Metrics:')
for k, v in metrics_data['overall_metrics'].items():
    print(f'  {k:20s}: {v:.4f}')


## 2. Per-Class Performance Breakdown

In [ ]:
per_class_df = pd.DataFrame(metrics_data['per_class_metrics']).T
print(per_class_df)


## 3. Confusion Matrix

In [ ]:
cm = np.array(metrics_data['confusion_matrix']['matrix'])
plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap=plt.cm.Blues)
plt.title('Test Set Confusion Matrix (UNSW-NB15)')
plt.xticks(range(len(CLASSES)), CLASSES, rotation=45)
plt.yticks(range(len(CLASSES)), CLASSES)
thresh = cm.max() / 2.0
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        plt.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                 color='white' if cm[i, j] > thresh else 'black')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.colorbar()
plt.tight_layout()
plt.show()


## 4. Discrepancy Inspection (False Positives and False Negatives)

In [ ]:
test_df = pd.read_csv('../../data/processed/clean_test.csv')
fp_indices = metrics_data['discrepancies']['false_positive_sample_indices']
fn_indices = metrics_data['discrepancies']['false_negative_sample_indices']

print(f"Total False Positives: {metrics_data['discrepancies']['false_positive_count']:,}")
print(f"Total False Negatives: {metrics_data['discrepancies']['false_negative_count']:,}")

print('\nTop 5 False Positive Rows (Normal flows misclassified as Attack):')
print(test_df.iloc[fp_indices[:5]][['proto', 'service', 'state', 'dur', 'sbytes', 'dbytes', 'rate', 'attack_cat']])

print('\nTop 5 False Negative Rows (Attacks misclassified as Normal):')
print(test_df.iloc[fn_indices[:5]][['proto', 'service', 'state', 'dur', 'sbytes', 'dbytes', 'rate', 'attack_cat']])


## 5. Architectural Findings
- **High Attack Sensitivity**: The model successfully flags 95.38% of all true malicious flows.
- **Benign Traffic Confusion**: 22.77% of normal flows are flagged as attacks, primarily due to high packet rates or unfamiliar state transitions.
- **Hard Categories**: As expected from the data audit, `Analysis` and `Backdoor` are the most difficult to distinguish due to small support (< 1.2%) and behavioral similarity to common web exploits.
